# Underwater Vessel Noise — minimal starter

Load the two challenge CSVs and create a valid constant-baseline submission using **pandas** and **NumPy**. No model training is needed. The optional external-audio example at the end uses **SciPy** when called.

Place `Test.csv` and `SampleSubmission.csv` beside this notebook. In Colab, upload them using the Files sidebar. Change `DATA_DIR` below if needed, then run the cells in order. The separate `South_African_Real_Acoustics.zip` provides unlabelled training measurements.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path(".")
OUTPUT_FILE = Path("submission.csv")

## 1. Load the data

Each test row is one **15-minute observation window**, identified by `ID`. Its 375 acoustic columns represent 15 minute positions ×25 frequency bands, ordered from minute 1 to minute 15 and from low to high frequency within each minute.

For example, `m01_TOL_31.6` is the band centred at 31.6 Hz at minute position 1; `m15_TOL_7943.3` is the band centred at 7943.3 Hz at minute position 15. Values are band levels in the source-export dB convention. Blank cells are missing observations, not silence or zero energy. Minute positions are relative to the window, not observation timestamps.

The bands use a base-10 third-octave (deci-decade) grid. The stated frequencies are band centres. Absolute pressure calibration of the original South African exports has not been independently verified.

In [ ]:
test = pd.read_csv(DATA_DIR / "Test.csv", dtype={"ID": str})
sample = pd.read_csv(DATA_DIR / "SampleSubmission.csv", dtype={"ID": str})

assert test["ID"].is_unique and sample["ID"].is_unique
assert test["ID"].notna().all() and sample["ID"].notna().all()
assert set(test["ID"]) == set(sample["ID"])
print("Test:", test.shape, "Sample submission:", sample.shape)

In [ ]:
feature_columns = [column for column in test.columns if column != "ID"]
assert len(feature_columns) == 15 * 25

# Array axes: example, minute position, frequency band.
acoustics = test[feature_columns].to_numpy(dtype=np.float32).reshape(-1, 15, 25)
valid_mask = np.isfinite(acoustics)
print("Acoustic array:", acoustics.shape)
print("Missing cells:", int((~valid_mask).sum()))

## 2. Make a constant baseline

Predict `share_added_vessel = 0.1` and `score_added_vessel = 0.5` for every window. These are arbitrary starting values, not learned estimates.

`share_added_vessel` estimates the fraction of mixture energy contributed by a controlled vessel-passage addition. It is an energy fraction, not a vessel-presence probability. The platform displays **1 − RMSE** on this output: **higher is better**, and a perfect prediction scores 1.

Real backgrounds can already contain vessels, and passage references may contain ambient sound. This target does not describe all natural vessel noise, ship count or size, or a calibrated pressure map. Replace the constant with inference from your own model trained on permitted data. Do not train or calibrate on test inputs.

The required submission has `ID`, `share_added_vessel` and `score_added_vessel`. The evidence score describes support for the controlled addition. ROC AUC on that output is diagnostic only; its platform weight is zero, so it does not affect ranking. It is not a calibrated probability of natural vessel presence.

In [ ]:
submission = sample.copy()
submission["share_added_vessel"] = 0.1
submission["score_added_vessel"] = 0.5

## 3. Validate and save

Keep the original IDs and submit exactly `ID,share_added_vessel,score_added_vessel`. Both predictions must be finite numbers between 0 and 1. Upload the resulting `submission.csv` to Zindi. The sample contains placeholders; test answers are not supplied.

In [ ]:
required_columns = ["ID", "share_added_vessel", "score_added_vessel"]
assert submission.columns.tolist() == required_columns
assert len(submission) == len(test)
assert submission["ID"].is_unique
assert set(submission["ID"]) == set(test["ID"])

predictions = submission[required_columns[1:]].to_numpy(dtype=float)
assert np.isfinite(predictions).all()
assert ((predictions >= 0) & (predictions <= 1)).all()

submission.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {OUTPUT_FILE}: {len(submission)} rows")

## Sources and data-use notices

Adapted vessel-passage components derive from **VTUAD: An AIS-Auto-Labeled Vessel-Type Underwater Acoustic Dataset Reconstructed from Ocean Networks Canada Archives**, v1.3.2, by Hao Yuan, Wenbo Wang, Tian Li, Guici Chen, Lingjiang Zeng and Sixian Liu. Source: https://doi.org/10.5281/zenodo.23037895. Licence: **CC BY 4.0**, https://creativecommons.org/licenses/by/4.0/.

Upstream acoustic data: **Ocean Networks Canada, Oceans 3.0**, https://www.oceannetworks.ca/; repository identifier: https://doi.org/10.17616/R3RW43.

Changes: audio was converted into minute-level frequency-band energy summaries; absolute digital gain was discarded; time trajectories, relative acoustic channel and energy were varied before combination with measured South African backgrounds. The supplied CSV arranges each window into a single row. No original external audio is redistributed in these files.

South African measurements were supplied through the **South African Underwater Radiated Noise consortium and data holders** for this challenge. Their applicable challenge data-use terms remain in force. The CC BY licence above covers the adapted VTUAD components; it does not assign that licence to the South African measurements or imply provider endorsement.

## Optional: preparing features from external audio

This reference conversion produces minute-level band mean-square values. It is not a verified reconstruction of the original South African export process.

Use one waveform channel and the actual audio sample rate. Convert digital samples or voltage to pressure only when the recording's ADC scaling, gain and hydrophone calibration are available. The output units follow the input: pressure in Pa gives Pa²; uncalibrated digital samples do not give absolute acoustic pressure. Frequency-dependent sensitivity may require an additional correction.

The function below uses non-overlapping 2-second segments, removes each segment's mean, estimates a one-sided PSD with a Hann window, integrates over each band, then averages 30 segments in linear units for each complete minute. Unsupported frequency bands are NaN, not zero. A trailing incomplete minute is omitted.

For calibrated Pa², a band level in dB re 1 µPa² is `10 * log10(band_power / 1e-12)`. Do not use that reference for uncalibrated samples. Average in linear units before taking logarithms. These summaries are not PSD per Hz or sound exposure level (SEL). Matching this representation does not establish physical equivalence between recorders.

In [ ]:
def minute_band_power(samples, sample_rate_hz):
    """One-channel audio -> complete-minute mean-square values, shape (minutes, 25)."""
    from scipy.signal import periodogram

    samples = np.asarray(samples, dtype=np.float64)
    if samples.ndim != 1 or not np.isfinite(samples).all():
        raise ValueError("Use one channel with finite samples.")
    fs = float(sample_rate_hz)
    if not np.isfinite(fs) or fs <= 0 or not fs.is_integer():
        raise ValueError("Use the actual integer sample rate from the recording.")
    segment_size = int(2 * fs)
    minutes = samples.size // (30 * segment_size)
    if minutes == 0:
        raise ValueError("At least one complete minute is needed.")
    segments = samples[:minutes * 30 * segment_size].reshape(-1, segment_size)
    frequency, psd = periodogram(
        segments, fs=fs, window="hann", detrend="constant",
        return_onesided=True, scaling="density", axis=1,
    )
    centres = 10.0 ** (1.5 + 0.1 * np.arange(25))
    power = np.full((len(segments), 25), np.nan)
    df = frequency[1] - frequency[0]
    for band, centre in enumerate(centres):
        low, high = centre / 10**0.05, centre * 10**0.05
        bins = (frequency >= low) & (frequency < high)
        if high <= fs / 2 and bins.any():
            power[:, band] = psd[:, bins].sum(axis=1) * df
    return power.reshape(minutes, 30, 25).mean(axis=1)
